# 📖 Notebook 1: Indexing and Full-Text Search

In this notebook, you'll learn the fundamentals of Elasticsearch — how to store data, retrieve it, and search through it using full-text queries.

## Learning Objectives

By the end of this notebook, you'll understand:
- What documents and indices are in Elasticsearch
- How to create, read, update, and delete documents (CRUD)
- How full-text search works with `match` queries
- How to combine conditions using `bool` queries
- How pagination and sorting work

## 🛠️ Setup

Start the infrastructure first:

```bash
cd deep-dives/elasticsearch
docker-compose up -d
```

### Visualization Tools

- **Kibana** (Elasticsearch GUI): http://localhost:5601  
  Go to **Management → Dev Tools** to run queries interactively.

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
from elasticsearch import Elasticsearch
import json
import time

# Connect to Elasticsearch running in Docker
es = Elasticsearch("http://localhost:9200")

# Verify the connection
info = es.info()
print(f"✅ Connected to Elasticsearch {info['version']['number']}")
print(f"   Cluster name: {info['cluster_name']}")
print(f"   Cluster UUID: {info['cluster_uuid']}")

## 📦 What Are Documents and Indices?

Think of Elasticsearch like a library:

| Library Concept | Elasticsearch Concept | Description |
|----------------|----------------------|-------------|
| A bookshelf section | **Index** | A collection of similar items |
| A single book | **Document** | One item (a JSON object) |
| The book's ISBN | **Document ID** | A unique identifier |
| The book's details | **Fields** | Key-value pairs (title, author, price) |

An **index** is like a database table — it holds a collection of **documents** (JSON objects).

Let's create a bookstore index and add some books!

## 🏗️ Creating an Index

Before we can store documents, we need to create an index. This is like creating
a table in SQL — it tells Elasticsearch what kind of data we'll store.

When creating an index, we can specify:
- **Settings** — how many shards and replicas (we'll cover this in Notebook 4)
- **Mappings** — the schema (field names and types, covered in Notebook 2)

For now, we'll let Elasticsearch figure out the schema automatically ("dynamic mapping").

In [ ]:
# Delete the index if it already exists (so we start fresh each time)
INDEX_NAME = "books"

if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)
    print(f"🗑️  Deleted existing '{INDEX_NAME}' index")

# Create a new index with 1 shard and 0 replicas (good for local dev)
es.indices.create(
    index=INDEX_NAME,
    settings={
        "number_of_shards": 1,
        "number_of_replicas": 0
    }
)
print(f"✅ Created '{INDEX_NAME}' index")

## ➕ Adding Documents (Indexing)

"Indexing" a document means storing it in Elasticsearch so it becomes searchable.

Each document is a JSON object. Elasticsearch will automatically detect the field
types (string, number, date) — this is called **dynamic mapping**.

In [ ]:
# Our sample bookstore data
books = [
    {
        "title": "The Great Gatsby",
        "author": "F. Scott Fitzgerald",
        "description": "A novel about the American Dream in the Jazz Age",
        "price": 9.99,
        "publish_date": "1925-04-10",
        "categories": ["Classic", "Fiction"],
        "in_stock": True,
        "rating": 4.5
    },
    {
        "title": "To Kill a Mockingbird",
        "author": "Harper Lee",
        "description": "A novel about racial injustice in the American South",
        "price": 12.99,
        "publish_date": "1960-07-11",
        "categories": ["Classic", "Fiction"],
        "in_stock": True,
        "rating": 4.8
    },
    {
        "title": "1984",
        "author": "George Orwell",
        "description": "A dystopian novel about totalitarianism and surveillance",
        "price": 11.99,
        "publish_date": "1949-06-08",
        "categories": ["Classic", "Dystopian", "Fiction"],
        "in_stock": True,
        "rating": 4.7
    },
    {
        "title": "Clean Code",
        "author": "Robert C. Martin",
        "description": "A handbook of agile software craftsmanship for writing better code",
        "price": 29.99,
        "publish_date": "2008-08-01",
        "categories": ["Technology", "Programming"],
        "in_stock": True,
        "rating": 4.3
    },
    {
        "title": "Designing Data-Intensive Applications",
        "author": "Martin Kleppmann",
        "description": "A guide to the big ideas behind reliable, scalable, and maintainable systems",
        "price": 39.99,
        "publish_date": "2017-03-16",
        "categories": ["Technology", "System Design"],
        "in_stock": False,
        "rating": 4.9
    },
    {
        "title": "The Pragmatic Programmer",
        "author": "David Thomas and Andrew Hunt",
        "description": "Your journey to mastery in software development",
        "price": 34.99,
        "publish_date": "1999-10-20",
        "categories": ["Technology", "Programming"],
        "in_stock": True,
        "rating": 4.6
    },
    {
        "title": "Great Expectations",
        "author": "Charles Dickens",
        "description": "A coming-of-age story set in Victorian England about ambition and love",
        "price": 7.99,
        "publish_date": "1861-08-01",
        "categories": ["Classic", "Fiction"],
        "in_stock": True,
        "rating": 4.2
    },
    {
        "title": "Brave New World",
        "author": "Aldous Huxley",
        "description": "A dystopian novel about a technologically advanced future society",
        "price": 10.99,
        "publish_date": "1932-01-01",
        "categories": ["Classic", "Dystopian", "Fiction"],
        "in_stock": True,
        "rating": 4.4
    }
]

# Index each book — Elasticsearch assigns an auto-generated ID
for i, book in enumerate(books):
    result = es.index(index=INDEX_NAME, id=i+1, document=book)
    print(f"  Indexed: {book['title']:<45} → ID: {result['_id']}, version: {result['_version']}")

# Elasticsearch indexes are "near real-time": new documents become searchable
# after a refresh (default: every 1 second). Let's force a refresh now.
es.indices.refresh(index=INDEX_NAME)

print(f"\n✅ Indexed {len(books)} books into '{INDEX_NAME}'")

## 📖 Reading a Document by ID

Just like a SQL `SELECT * FROM books WHERE id = 1`, you can fetch a single
document by its ID. This is an O(1) lookup — very fast!

In [ ]:
# Get a single document by ID
doc = es.get(index=INDEX_NAME, id=1)

print("📖 Document with ID=1:")
print(f"   Index:   {doc['_index']}")
print(f"   ID:      {doc['_id']}")
print(f"   Version: {doc['_version']}")
print(f"   Source:  {json.dumps(doc['_source'], indent=2)}")

## ✏️ Updating a Document

You can update documents in two ways:

1. **Full replacement** — send the entire document (PUT)
2. **Partial update** — send only the fields you want to change

Each update increments the document's `_version` field. Elasticsearch uses this
for **optimistic concurrency control** — if two people try to update the same
document at the same time, the second update will know about the conflict.

In [ ]:
# Partial update: change just the price of "The Great Gatsby"
result = es.update(
    index=INDEX_NAME,
    id=1,
    doc={"price": 14.99}  # only update the price field
)
print(f"✏️  Updated document 1 → new version: {result['_version']}")

# Verify the update
updated_doc = es.get(index=INDEX_NAME, id=1)
print(f"   New price: ${updated_doc['_source']['price']}")
print(f"   Title still intact: {updated_doc['_source']['title']}")

## 🗑️ Deleting a Document

Fun fact: Elasticsearch doesn't actually delete the data immediately! It marks
the document as "deleted" (a soft delete), and the data is cleaned up later
during a segment merge operation. More on this in Notebook 4.

In [ ]:
# Let's add a book and then delete it
es.index(index=INDEX_NAME, id=99, document={
    "title": "Temporary Book",
    "author": "Nobody",
    "price": 0.00
})
es.indices.refresh(index=INDEX_NAME)

# Count before delete
count_before = es.count(index=INDEX_NAME)["count"]
print(f"📊 Books before delete: {count_before}")

# Delete the temporary book
es.delete(index=INDEX_NAME, id=99)
es.indices.refresh(index=INDEX_NAME)

# Count after delete
count_after = es.count(index=INDEX_NAME)["count"]
print(f"📊 Books after delete:  {count_after}")
print(f"🗑️  Removed {count_before - count_after} document(s)")

---

## 🔍 Full-Text Search

This is where Elasticsearch really shines! Unlike a SQL `WHERE title = 'Great'`,
Elasticsearch can:

1. **Tokenize** the text — break "The Great Gatsby" into ["the", "great", "gatsby"]
2. **Match** your query against those tokens
3. **Score** each document by how relevant it is to your query

The simplest search is a **match** query — it finds documents where a field
contains the words you're looking for.

In [ ]:
def print_search_results(results, show_score=True):
    """Helper to display search results nicely."""
    hits = results["hits"]["hits"]
    total = results["hits"]["total"]["value"]
    took = results["took"]
    print(f"Found {total} result(s) in {took}ms:\n")

    for hit in hits:
        source = hit["_source"]
        score = f" (score: {hit['_score']:.4f})" if show_score and hit['_score'] else ""
        print(f"  📗 {source['title']}{score}")
        print(f"     Author: {source.get('author', 'N/A')}")
        print(f"     Price: ${source.get('price', 'N/A')}")
        if 'description' in source:
            print(f"     Description: {source['description']}")
        print()

In [ ]:
# Simple match query: find books with "great" in the title
print("🔍 Search: books with 'great' in the title")
print("=" * 50)

results = es.search(
    index=INDEX_NAME,
    query={
        "match": {
            "title": "great"
        }
    }
)

print_search_results(results)
print("💡 Notice: both 'The Great Gatsby' and 'Great Expectations' matched!")
print("   Elasticsearch automatically lowercased 'Great' → 'great' for matching.")

In [ ]:
# Search across a different field: search descriptions for "dystopian"
print("🔍 Search: books with 'dystopian' in the description")
print("=" * 55)

results = es.search(
    index=INDEX_NAME,
    query={
        "match": {
            "description": "dystopian"
        }
    }
)

print_search_results(results)

## 🔗 Bool Queries: Combining Conditions

Real searches usually have multiple conditions. The `bool` query lets you combine
them using:

| Clause | SQL Equivalent | Effect on Score |
|--------|---------------|----------------|
| `must` | AND | Yes — boosts matching docs |
| `should` | OR | Yes — boosts matching docs |
| `must_not` | NOT | No — just excludes |
| `filter` | AND (but no scoring) | No — just includes/excludes |

The difference between `must` and `filter` is important:
- `must` affects the **relevance score** (how documents are ranked)
- `filter` is a yes/no check (faster because Elasticsearch can cache it)

In [ ]:
# Bool query: fiction books under $12
print("🔍 Search: fiction books under $12")
print("=" * 40)

results = es.search(
    index=INDEX_NAME,
    query={
        "bool": {
            "must": [
                {"match": {"categories": "Fiction"}}  # must be fiction
            ],
            "filter": [
                {"range": {"price": {"lte": 12}}}    # price filter (no scoring)
            ]
        }
    }
)

print_search_results(results)

In [ ]:
# Bool query with must_not: classic books that are NOT dystopian
print("🔍 Search: classic books that are NOT dystopian")
print("=" * 50)

results = es.search(
    index=INDEX_NAME,
    query={
        "bool": {
            "must": [
                {"match": {"categories": "Classic"}}
            ],
            "must_not": [
                {"match": {"categories": "Dystopian"}}
            ]
        }
    }
)

print_search_results(results)

In [ ]:
# Bool query with should: prefer books about "software" OR "code"
print("🔍 Search: tech books (boost those about 'software' or 'code')")
print("=" * 60)

results = es.search(
    index=INDEX_NAME,
    query={
        "bool": {
            "must": [
                {"match": {"categories": "Technology"}}
            ],
            "should": [
                {"match": {"description": "software"}},
                {"match": {"description": "code"}}
            ]
        }
    }
)

print_search_results(results)
print("💡 'should' doesn't exclude results — it boosts those that match.")
print("   Books mentioning 'software' or 'code' get higher scores.")

## 🔤 Multi-Match: Search Across Multiple Fields

What if you want to search for a term across the title AND description at the
same time? The `multi_match` query does exactly that.

In [ ]:
# Search for "american" across both title and description
print("🔍 Multi-match: 'american' in title or description")
print("=" * 55)

results = es.search(
    index=INDEX_NAME,
    query={
        "multi_match": {
            "query": "american",
            "fields": ["title", "description"]
        }
    }
)

print_search_results(results)

## 📄 Sorting and Pagination

By default, Elasticsearch sorts results by **relevance score** (`_score`).
But you can sort by any field — price, date, rating, etc.

### Pagination

Elasticsearch uses `from` + `size` for simple pagination:
- `from` = starting position (0-based)
- `size` = how many results to return

⚠️ **Warning**: This gets slow for deep pages (from > 10,000) because
Elasticsearch must sort ALL preceding documents first. For deep pagination,
use `search_after` instead (we'll show this too).

In [ ]:
# Sort all books by price (cheapest first)
print("📄 All books sorted by price (ascending)")
print("=" * 45)

results = es.search(
    index=INDEX_NAME,
    query={"match_all": {}},
    sort=[{"price": "asc"}],
    size=5  # only show first 5
)

# When sorting by a field, _score is null (we're not ranking by relevance)
print_search_results(results, show_score=False)

In [ ]:
# Pagination example: get results in pages of 3
PAGE_SIZE = 3

print("📄 Pagination Demo: 3 books per page, sorted by price")
print("=" * 55)

for page_num in range(1, 4):  # pages 1, 2, 3
    from_offset = (page_num - 1) * PAGE_SIZE
    results = es.search(
        index=INDEX_NAME,
        query={"match_all": {}},
        sort=[{"price": "asc"}],
        from_=from_offset,
        size=PAGE_SIZE
    )

    hits = results["hits"]["hits"]
    if not hits:
        break

    print(f"\n--- Page {page_num} (from={from_offset}, size={PAGE_SIZE}) ---")
    for hit in hits:
        src = hit["_source"]
        print(f"  ${src['price']:<8} {src['title']}")

## 🧩 Bulk Indexing

So far we've been indexing documents one at a time. In production, you'd use
the **bulk API** to index thousands of documents in a single request.
This is *much* faster because it reduces network round trips.

In [ ]:
from elasticsearch.helpers import bulk

# More books to bulk-index
more_books = [
    {"title": "Sapiens", "author": "Yuval Noah Harari", "description": "A brief history of humankind from the Stone Age to the present", "price": 15.99, "categories": ["Non-Fiction", "History"], "rating": 4.6, "in_stock": True},
    {"title": "Atomic Habits", "author": "James Clear", "description": "Tiny changes remarkable results for building good habits", "price": 13.99, "categories": ["Non-Fiction", "Self-Help"], "rating": 4.8, "in_stock": True},
    {"title": "The Hitchhiker's Guide to the Galaxy", "author": "Douglas Adams", "description": "A comedic science fiction series about the absurdity of the universe", "price": 8.99, "categories": ["Science Fiction", "Comedy"], "rating": 4.7, "in_stock": True},
    {"title": "Dune", "author": "Frank Herbert", "description": "An epic science fiction novel about politics and survival on a desert planet", "price": 11.49, "categories": ["Science Fiction", "Fiction"], "rating": 4.6, "in_stock": True},
    {"title": "Introduction to Algorithms", "author": "Thomas Cormen", "description": "A comprehensive textbook on computer science algorithms and data structures", "price": 74.99, "categories": ["Technology", "Textbook"], "rating": 4.4, "in_stock": False},
]

# Prepare bulk actions
actions = [
    {
        "_index": INDEX_NAME,
        "_source": book
    }
    for book in more_books
]

# Bulk index
success, errors = bulk(es, actions)
es.indices.refresh(index=INDEX_NAME)

total = es.count(index=INDEX_NAME)["count"]
print(f"✅ Bulk indexed {success} documents ({len(errors)} errors)")
print(f"📊 Total documents in '{INDEX_NAME}': {total}")

## 🧪 Exercises

Try these yourself! Modify the queries above or write new ones:

1. **Find all science fiction books** — use a `match` query on the `categories` field
2. **Find books by a specific author** — search for "Orwell" in the `author` field
3. **Find in-stock books rated above 4.5** — combine `filter` clauses in a `bool` query
4. **Sort books by rating (highest first)** — use the `sort` parameter
5. **Search for "systems" across all text fields** — use a `multi_match` query

In [ ]:
# Exercise space — try your queries here!

# Example: Find all science fiction books
# results = es.search(
#     index=INDEX_NAME,
#     query={
#         "match": {"categories": "Science Fiction"}
#     }
# )
# print_search_results(results)

## 🎯 Key Takeaways

1. **Documents** are JSON objects stored in an **index** (like rows in a table)
2. **Indexing** = storing a document; **Searching** = querying for documents
3. `match` queries do **full-text search** — they tokenize text and find matches
4. `bool` queries combine conditions: `must` (AND), `should` (OR), `must_not` (NOT), `filter` (AND without scoring)
5. Results are ranked by **relevance score** — how well they match your query
6. Use `from`/`size` for simple pagination, but beware of deep pagination
7. The **bulk API** is essential for production — always batch your writes

**Next up**: Notebook 2 covers **Analyzers and Mappings** — how Elasticsearch
processes text and how you can control it for better search results.